In [2]:
import re
import os
import pandas as pd
import numpy as np
from Bio import SeqIO
from datetime import datetime

In [61]:
US_COUNTIES = "./metadata/detections_data/2024_Gaz_counties_national.txt"
CAN_COUNTIES = "./metadata/detections_data/georef-canada-census-subdivision.csv"

In [62]:
us_counties = pd.read_csv(US_COUNTIES, sep = "\t")
can_counties = pd.read_csv(CAN_COUNTIES, sep = ",")

In [63]:
us_counties.head()

,USPS,GEOID,ANSICODE,NAME,ALAND,AWATER,ALAND_SQMI,AWATER_SQMI,INTPTLAT,INTPTLONG
0,AL,1001,161526,Autauga County,1539631459,25677536,594.455,9.914,32.532237,-86.646440
1,AL,1003,161527,Baldwin County,4117781416,1132830835,1589.884,437.388,30.659218,-87.746067
2,AL,1005,161528,Barbour County,2292160151,50523213,885.008,19.507,31.870253,-85.405104
3,AL,1007,161529,Bibb County,1612188713,9572302,622.470,3.696,33.015893,-87.127148
4,AL,1009,161530,Blount County,1670259099,14860281,644.891,5.738,33.977358,-86.566440


In [64]:
can_counties.head()

,Geo Point,Geo Shape,Year,Official Code Province / Territory,Official Name Province / Territory (English),Official Code Census Division,Official Name Census Division (English),Official Code Census Subdivision,Official Name Census Subdivision (English),Iso 3166-3 Area Code,Type (English / French),Official Name Province / Territory (French),Official Name Census Division (French),Official Name Census Subdivision (French)
0,"50.723845002186756, -121.30679925043623","{""coordinates"":[[[-121.31245432664745,50.70867...",2021.0,59.0,British Columbia,5933.0,Thompson-Nicola,5933845.0,105 Mile Post 2,CAN,Indian reserve / Réserve indienne,Colombie-Britannique,Thompson-Nicola,105 Mile Post 2
1,"56.34611197620165, -94.70953396907967","{""coordinates"":[[[-94.71095545416914,56.345598...",2021.0,46.0,Manitoba,4623.0,Division No. 23,4623023.0,A Kwis Ki Mahka,CAN,Indian reserve / Réserve indienne,Manitoba,Division No. 23,A Kwis Ki Mahka
2,"50.73626410074064, -108.75743053658263","{""coordinates"":[[[-108.76313192049199,50.74094...",2021.0,47.0,Saskatchewan,4708.0,Division No. 8,4708048.0,Abbey,CAN,Village,Saskatchewan,Division No. 8,Abbey
3,"49.06862082049634, -122.28387793346447","{""coordinates"":[[[-122.3384169476604,49.109050...",2021.0,59.0,British Columbia,5909.0,Fraser Valley,5909052.0,Abbotsford,CAN,City,Colombie-Britannique,Fraser Valley,Abbotsford
4,"45.04013163033006, -72.6637530134263","{""coordinates"":[[[-72.63257593986745,45.014585...",2021.0,24.0,Quebec,2446.0,Brome-Missisquoi,2446005.0,Abercorn,CAN,Village,Québec,Brome-Missisquoi,Abercorn


In [65]:
# Adjust the US Counties dataset to only the needed columns

# Drop GEOID, ANSICODE, ALAND, ALAND_SQMI, AWATER, AWATER_SQMI
# Keep USPS, NAME, INTPLAT, INTPTLONG

us_counties.columns = us_counties.columns.str.replace(' ', '') # This because INTPTLONG actually has a weird long space after it?

us_counties_columns_keep = ["USPS", "NAME", "INTPTLAT", "INTPTLONG"]

us_counties_subset = us_counties.loc[0:, us_counties_columns_keep]

# Rename the columns

us_county_column_mapper = {"USPS":"state", "NAME":"county", "INTPTLAT":"centroid_lat", "INTPTLONG":"centroid_long"}

us_counties_subset_renamed = us_counties_subset.rename(us_county_column_mapper, axis = 1)

# View

print(len(us_counties))
print(len(us_counties_subset_renamed)) # There are 3222 US counties

3222
3222


In [66]:
# Adjust the Canadian Counties dataset to only the needed columns

# Keep Geo Point, Official Name Province / Territory (English), Official Name Census Subdivision (English)

can_counties_columns_keep = ["Geo Point", 
                            "Official Name Province / Territory (English)", 
                            "Official Name Census Division (English)",
                            "Official Name Census Subdivision (English)"
                           ]

can_counties_subset = can_counties.loc[0:, can_counties_columns_keep]

# Rename the columns

can_county_column_mapper = {"Official Name Province / Territory (English)":"state", 
                            "Official Name Census Division (English)":"division",
                            "Official Name Census Subdivision (English)":"county", 
                            "Geo Point":"lat_long"}

can_counties_subset_renamed = can_counties_subset.rename(can_county_column_mapper, axis = 1)

# View
print(len(can_counties))
print(len(can_counties_subset_renamed)) # There are 5162 Canadian census subdivisions

5162
5162


In [67]:
state_abbr_dict = {"AL":"alabama",
                   "AK":"alaska",
                   "AZ":"arizona",
                   "AR":"arkansas",
                   "CA":"california",
                   "CO":"colorado",
                   "CT":"connecticut",
                   "DC":"dc",
                   "DE":"delaware",
                   "FL":"florida",
                   "GA":"georgia",
                   "HI":"hawaii",
                   "ID":"idaho",
                   "IL":"illinois",
                   "IN":"indiana",
                   "IA":"iowa",
                   "KS":"kansas",
                   "KY":"kentucky",
                   "LA":"louisiana",
                   "ME":"maine",
                   "MD":"maryland",
                   "MA":"massachusetts",
                   "MI":"michigan",
                   "MN":"minnesota",
                   "MS":"mississippi",
                   "MO":"missouri",
                   "MT":"montana",
                   "NE":"nebraska",
                   "NV":"nevada",
                   "NH":"newhampshire",
                   "NJ":"newjersey",
                   "NM":"newmexico",
                   "NY":"newyork",
                   "NC":"northcarolina",
                   "ND":"northdakota",
                   "OH":"ohio",
                   "OK":"oklahoma",
                   "OR":"oregon",
                   "PA":"pennsylvania",
                   "PR":"puertorico",
                   "RI":"rhodeisland",
                   "SC":"southcarolina",
                   "SD":"southdakota",
                   "TN":"tennessee",
                   "TX":"texas",
                   "UT":"utah",
                   "VT":"vermont",
                   "VA":"virginia",
                   "WA":"washington",
                   "WV":"westvirginia",
                   "WI":"wisconsin",
                   "WY":"wyoming"
                       } 

In [68]:
# Function: some issues in the county column

def county_column(string):
    
    # Initialize new_string to the original string
    new_string = string

    # Remove 'County' (common in most states)
    new_string = new_string.replace("County", "")
    
    # Remove 'Parish' (used in Louisiana)
    new_string = new_string.replace("Parish", "")
    
    # Remove 'Municipio' (used in Puerto Rico)
    new_string = new_string.replace("Municipio", "")

    # Alaska is weird
    alaska_counties_dict = {"Aleutians East Borough":"Aleutians East",
                            "Aleutians West Census Area":"Aleutians West",
                            "Anchorage Municipality":"Anchorage",
                            "Bethel Census Area":"Bethel",
                            #"Bristol Bay Borough":,
                            "Chugach Census Area":"Chugach",
                            #"Copper River Census Area":, does this map to valdez-cordova then?
                            #"Denali Borough":,
                            "Dillingham Census Area":"Dillingham",
                            "Fairbanks North Star Borough":"Fairbanks North Star",
                            "Haines Borough":"Haines",
                            #"Hoonah-Angoon Census Area":,
                            "Juneau City and Borough":"Juneau",
                            "Kenai Peninsula Borough":"Kenai Peninsula",
                            "Ketchikan Gateway Borough":"Ketchikan Gateway",
                            #"Kodiak Island Borough":,
                            "Kusilvak Census Area":"Kusilvak",
                            #"Lake and Peninsula Borough":,
                            "Matanuska-Susitna Borough":"Matanuska-Susitna",
                            "Nome Census Area":"Nome",
                            "North Slope Borough":"North Slope",
                            "Northwest Arctic Borough":"Northwest Arctic",
                            #"Petersburg Borough":,
                            #"Prince of Wales-Hyder Census Area":,
                            "Sitka City and Borough":"Sitka",
                            #"Skagway Municipality":,
                            "Southeast Fairbanks Census Area":"Southeast Fairbanks",
                            #"Wrangell City and Borough":,
                            #"Yakutat City and Borough":,
                            "Yukon-Koyukuk Census Area":"Yukon-Koyukuk"
                           }
    for census_name, detections_name in alaska_counties_dict.items():
        new_string = new_string.replace(census_name, detections_name)

    return(new_string)

In [69]:
# Function: reorganize us_counties_subset_renamed

def fix_uscounties_dataframe(df):

    df_final = df.copy()

    # First, add a column with full state name
    df_final["full_state"] = df_final["state"].map(state_abbr_dict)

    # Next, fix the county column
    df_final["county_fixed"] = df_final["county"].apply(county_column)
    df_final["county_fixed"] = df_final["county_fixed"].str.rstrip()

    df_final["county_fixed"] = df_final["county_fixed"].str.replace(".", "")
    df_final["county_fixed"] = df_final["county_fixed"].str.replace(" ", "_")
    df_final["county_fixed"] = df_final["county_fixed"].str.replace("-", "_")
    df_final["county_fixed"] = df_final["county_fixed"].str.replace("'", "")
    df_final["county_fixed"] = df_final["county_fixed"].str.lower()

    # Finally, join lat and long into a new column and join state and county into a new column
    df_final["lat_long"] = df_final["centroid_lat"].astype(str) + ', ' + df_final["centroid_long"].astype(str)

    df_final["state_county"] = df_final["full_state"] + '_' + df_final["county_fixed"]

    return(df_final)

In [70]:
# Make the final us_counties dataframe

us_counties_final = fix_uscounties_dataframe(us_counties_subset_renamed)

In [71]:
us_counties_final.head()

,state,county,centroid_lat,centroid_long,full_state,county_fixed,lat_long,state_county
0,AL,Autauga County,32.532237,-86.646440,alabama,autauga,"32.532237, -86.64644",alabama_autauga
1,AL,Baldwin County,30.659218,-87.746067,alabama,baldwin,"30.659218, -87.746067",alabama_baldwin
2,AL,Barbour County,31.870253,-85.405104,alabama,barbour,"31.870253, -85.405104",alabama_barbour
3,AL,Bibb County,33.015893,-87.127148,alabama,bibb,"33.015893, -87.127148",alabama_bibb
4,AL,Blount County,33.977358,-86.566440,alabama,blount,"33.977358, -86.56644",alabama_blount


In [72]:
# Print the set of unique Canadian counties

print(set(can_counties_subset_renamed["county"]))

{'Hepburn', 'Cremona', 'La Vallee', 'Foremost', 'Amqui', 'Dead Point 5', 'East Moberly Lake 169', 'Port Moody', 'Sidney', 'Spintlum Flat 3', 'Swan Lake 7', 'Shoal Lake 34B2', 'Division No.  1, Subd. W', 'Napierville', 'Peace River E', 'Fortune', 'Aquaforte', 'Louis Bull 138B', 'Telkwa', 'Carmacks', 'Keys No. 303', 'Waterhen 45', 'Green Lake', 'Saint-Robert-Bellarmin', 'Nenagwas 12', 'Blood 148', "Sainte-Famille-de-l'Île-d'Orléans", 'Saint-Narcisse', 'Saint-Prosper-de-Champlain', 'Severn', 'Odanak', 'Hawkesbury', 'Tanakut 4', 'Springfield', 'Manitou Rapids 11', 'Dunnottar', 'Membertou 28B', 'Saint-Isidore-de-Clifton', 'Rushoon', 'Pleasantdale', 'Vauxhall', 'Little Hills 158B', 'Cariboo L', 'Digby', 'Big Quill No. 308', 'Saint-Octave-de-Métis', 'Hines Creek', 'Glendon', 'Toosey 1', 'Dawson Creek', 'Improvement District No. 24 Wood Buffalo', 'Keels', 'Belle-Rivière', 'Nain', 'Port Anson', 'Kuujjuarapik', 'Kasabonika Lake', 'Saug-A-Gaw-Sing 1', 'Improvement District No. 9 Banff', 'Winnipeg

In [73]:
can_counties_subset_renamed.head()

,lat_long,state,division,county
0,"50.723845002186756, -121.30679925043623",British Columbia,Thompson-Nicola,105 Mile Post 2
1,"56.34611197620165, -94.70953396907967",Manitoba,Division No. 23,A Kwis Ki Mahka
2,"50.73626410074064, -108.75743053658263",Saskatchewan,Division No. 8,Abbey
3,"49.06862082049634, -122.28387793346447",British Columbia,Fraser Valley,Abbotsford
4,"45.04013163033006, -72.6637530134263",Quebec,Brome-Missisquoi,Abercorn


In [74]:
can_county_symbols = {"  ":" ",
                     ". ":"_",
                     ", ":"_",
                     " - ":"_",
                     "--":"_",
                     " / ":"_",
                     "'":"",
                     #""b"":"b",
                     "(":"",
                     ")":"",
                     "!":"",
                     "&":"and",
                     "é":"e",
                     "è":"e",
                     "ò":"o",
                     "í":"i",
                     "ì":"i",
                     "à":"a",
                     "á":"a",
                     "ú":"u",
                     "ù":"u",
                     "â":"a",
                     "ô":"o",
                     "ê":"e",
                     "ï":"i",
                     "ü":"u",
                     "ë":"e",
                     "ç":"c",
#                      "Î":"i"
                    }

In [75]:
# Function: replace the weirder symbols and French lettering in the Canadian county names

def replace_symbols(text):
    if isinstance(text, str):
        for symbol, replacement in can_county_symbols.items():
            text = text.replace(symbol, replacement)
    return text

In [76]:
# Function: reorganize can_counties_subset_renamed

def fix_cancounties_dataframe(df):

    df_final = df.copy()

    # First, fix the "state" column (technically provinces)
    df_final["state_fixed"] = df_final["state"]
    df_final["state_fixed"] = df_final["state_fixed"].str.replace(" ", "")
    df_final["state_fixed"] = df_final["state_fixed"].str.lower()

    # Next, fix the county column
    df_final["county_fixed"] = df_final["county"]
    df_final["county_fixed"] = df_final["county_fixed"].str.rstrip()
    df_final["county_fixed"] = df_final["county_fixed"].str.replace("Î", "i") # have to change this outside of the function, otherwise it won't work
    df_final["county_fixed"] = df_final["county_fixed"].str.lower()


    df_final["county_fixed"] = df_final["county_fixed"].apply(replace_symbols)
    
    # Fix these outside of the dictionary because otherwise they will confuse things
    df_final["county_fixed"] = df_final["county_fixed"].str.replace(" ", "_")
    df_final["county_fixed"] = df_final["county_fixed"].str.replace("-", "_")
    df_final["county_fixed"] = df_final["county_fixed"].str.replace("/", "_")


    df_final["state_county"] = df_final["state_fixed"] + '_' + df_final["county_fixed"]

    return(df_final)

In [77]:
# Make the final can_counties dataframe

can_counties_final = fix_cancounties_dataframe(can_counties_subset_renamed)

In [78]:
# Trying to figure out the canadian CCS situation

# canadasCCSisweird = can_counties_final.drop(["lat_long", "state", "county", "state_fixed", "state_county"], axis = 1)
# canadasCCSisweird.to_csv("./metadata/detections_data/Canada-CCS.csv", sep = ",")

In [79]:
# Trying to figure out what to do with canadian duplicate counties

print(can_counties_final["state_county"].is_unique)
print(can_counties_final["state_county"].value_counts().loc[lambda x: x > 1])

# Get the rows that appear more than once
freq = can_counties_final["state_county"].value_counts()
# index of items that appear more than once
items = freq[freq>1].index 
more_than_1_df = can_counties_final[can_counties_final["state_county"].isin(items)]
#more_than_1_df.to_csv("./metadata/detections_data/Canada-duplicates.csv", sep = ",")

# 133 pairs (266 total) of matching state_county
# 96 pairs have the same division (72%)
# 37 pairs have different divisions (28%)

False
state_county
newbrunswick_simonds                2
newbrunswick_hillsborough           2
quebec_ivujivik                     2
quebec_notre_dame_du_bon_conseil    2
newbrunswick_drummond               2
                                   ..
quebec_lac_simon                    2
princeedwardisland_alberton         2
newbrunswick_harvey                 2
novascotia_bear_river_part_6        2
quebec_chisasibi                    2
Name: count, Length: 133, dtype: int64


In [104]:
# If there are duplicate counties, change lat/long to "unknown"
# Function: change duplicate counties to unknown lat/long

def duplicate_counties(df):

    # Add state_county that appears omore than once to a list
    freq = df["state_county"].value_counts()
    duplicates = freq[freq>1].index.tolist()

    # Change the lat and long of each row included in "items" to "unknown"
    df.loc[df["state_county"].isin(duplicates), "lat_long"] = "unknown"

    # Then deduplicate the repeats
    df_new = df.drop_duplicates(subset = "state_county", keep = "first", inplace = False)
    
    return df_new

In [105]:
can_counties_final_dupesunknown = duplicate_counties(can_counties_final)

# Check that the duplicates are gone
print(len(can_counties_final))
print(len(can_counties_final_dupesunknown))

# Check that the duplicated state_county say "unknown"
check = can_counties_final_dupesunknown.loc[can_counties_final_dupesunknown["state_county"].isin(items)]
check.head()

5162
5029


,lat_long,state,division,county,state_fixed,county_fixed,state_county
34,unknown,Quebec,Nord-du-Québec,Akulivik,quebec,akulivik,quebec_akulivik
49,unknown,Prince Edward Island,Prince,Alberton,princeedwardisland,alberton,princeedwardisland_alberton
55,unknown,British Columbia,Mount Waddington,Alert Bay,britishcolumbia,alert_bay,britishcolumbia_alert_bay
83,unknown,New Brunswick,Albert,Alma,newbrunswick,alma,newbrunswick_alma
172,unknown,Quebec,Nord-du-Québec,Aupaluk,quebec,aupaluk,quebec_aupaluk


In [106]:
# Concatenate both dataframes

keep = ["state_county", "lat_long"]

prep_us = us_counties_final.loc[0:, keep]
prep_can = can_counties_final_dupesunknown.loc[0:, keep]

NAm_counties = pd.concat([prep_us, prep_can], axis = 0, join = "outer")

In [107]:
print(len(NAm_counties))

8251


In [108]:
# Save this concatenated list of counties and their coordinates

NAm_counties.to_csv("./metadata/detections_data/NAm_counties.tsv", sep = "\t")